In [1]:
%matplotlib inline
import pandas as pd
import numpy as np

In [2]:
def slow_hamming_distance(s1, s2):
    if len(s1) != len(s2):
        print(s1,s2)
        raise ValueError("Strand lengths are not equal!")
    term_list = []
    for ch1,ch2 in zip(s1,s2):
        if ch1 == "N" or ch2 == "N":
            term_list.append(False)
        else:
            term_list.append(ch1 != ch2)
    result = sum(term_list)
    return result

In [6]:
# Add fiducial sequences to the 
# Import nanopore data
df_nanopore = pd.read_csv("/home/lag36/scratch/lag36/2024-08-26_lLAG8_Nanopore/lLAG8_final_df.tsv", sep="\t").drop(columns=["Unnamed: 0"])

In [ ]:
df_nanopore

In [8]:
barcode_list = df_nanopore["barcode"]

In [ ]:
df_nanopore.columns

In [10]:
barcode_fiducial_1 = '1'*30 # LAG540
barcode_fiducial_2 = '10'*15 # LAG550

fiducial_barcodes = [barcode_fiducial_1, barcode_fiducial_2]

In [ ]:
# Get hamming distance matrix
hamming_distance_matrix = np.zeros((len(fiducial_barcodes), len(fiducial_barcodes)))
for i, barcode_1 in enumerate(fiducial_barcodes):
    for j, barcode_2 in enumerate(fiducial_barcodes):
        hamming_distance_matrix[i,j] = slow_hamming_distance(barcode_1, barcode_2)

# Place infinity in the diagonal
np.fill_diagonal(hamming_distance_matrix, np.inf)

# Get the min hamming distances across all fiducials
min_ham_distance_fiducials = np.min(hamming_distance_matrix, axis=1).astype(int)

# Get the list of closest hamming distances for each barcode
closest_hamming_distance = np.zeros(len(fiducial_barcodes), dtype=int)
for i, barcode in enumerate(fiducial_barcodes):
    print(i, barcode)
    closest_hamming_distance[i] = np.min([df_nanopore.apply(lambda x: slow_hamming_distance(x["barcode"], barcode), axis=1).min(), min_ham_distance_fiducials[i]])

# Add an entry for the fiducial sequences
df_nanopore_fiducials = (pd.concat([df_nanopore, pd.DataFrame({"barcode":fiducial_barcodes,"Closest Hamming Distance":closest_hamming_distance,
                                               "barcodeid": np.arange(len(fiducial_barcodes)) + df_nanopore['barcodeid'].max() + 1,
                                               "closest_match": closest_hamming_distance,
                                               "Category": ["fiducial"]*len(fiducial_barcodes)
                                               })])
                                               .reset_index(drop=True))


In [13]:
df_nanopore_fiducials.to_csv("/home/lag36/scratch/lag36/2024-08-26_lLAG8_Nanopore/lLAG8_final_df_fiducials.tsv", sep="\t")

In [ ]:
pd.read_csv("/home/lag36/scratch/lag36/2024-08-26_lLAG8_Nanopore/lLAG8_final_df.tsv",delimiter="\t",index_col=0)

In [16]:
df = pd.read_csv("/home/lag36/scratch/lag36/2024-08-26_lLAG8_Nanopore/lLAG8_final_df_fiducials.tsv",delimiter="\t",index_col=0)

In [ ]:
df['Closest Hamming Distance'].value_counts()/df['Closest Hamming Distance'].value_counts().sum()